In [17]:
import os, glob, cv2, numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

DATA_DIR = './data/images'
KELAS = ['night', 'day']    
BINS = 32
EXT = ('.jpg', '.jpeg', '.png', '.bmp')

In [18]:
def hist_feature(img):
  img = cv2.resize(img, (128, 128))
  feats = []
  for ch in range(3):
    h = cv2.calcHist([img], [ch], None, [BINS], [0, 256]).flatten()
    feats.append(h / h.sum())
  return np.concatenate(feats)

X, y = [], []
for label, kelas in enumerate(KELAS):
  pola = os.path.join(DATA_DIR, '**', kelas, '*')
  files = [f for f in glob.glob(pola, recursive=True) if f.lower().endswith(EXT)]
  print(f'{kelas}: {len(files)} file ditemukan')
  for p in files:
    img = cv2.imdecode(np.fromfile(p, dtype=np.uint8), cv2.IMREAD_COLOR)
    if img is not None:
      X.append(hist_feature(img)); y.append(label)

X, y = np.array(X), np.array(y)

night: 200 file ditemukan
day: 200 file ditemukan


In [19]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=42, stratify=y)

base_model = Pipeline([
  ('scaler', StandardScaler()),
  ('SVC', SVC(kernel='rbf'))
]).fit(X_train, y_train)

y_pred_test = base_model.predict(X_test)
print('akurasi test:', accuracy_score(y_test, y_pred_test))

akurasi test: 1.0


In [21]:
grid = GridSearchCV(
  Pipeline([
    ('scaler', StandardScaler()),
    ('SVC', SVC(kernel='rbf'))
  ]),
  param_grid={
    'SVC__C': [0.1, 1, 10, 100],
    'SVC__gamma': [0.0001, 0.001, 0.01, 0.1]
  }, cv=5, scoring='accuracy', n_jobs=-1
).fit(X_train, y_train)
print('param terbaik:', grid.best_params_)
print('akurasi terbaik:', grid.best_score_)
pred = grid.predict(X_test)
print('akurasi test setelah grid:', accuracy_score(y_test, pred))
print(classification_report(y_test, pred, target_names=['night', 'day']))

param terbaik: {'SVC__C': 1, 'SVC__gamma': 0.001}
akurasi terbaik: 0.990625
akurasi test setelah grid: 1.0
              precision    recall  f1-score   support

       night       1.00      1.00      1.00        40
         day       1.00      1.00      1.00        40

    accuracy                           1.00        80
   macro avg       1.00      1.00      1.00        80
weighted avg       1.00      1.00      1.00        80

